# GPU stage 2 -- CelebA on the full training split, with model selection

~2.5-3.5 h on an L4. Loads the Waterbirds records stage 1 wrote to Drive and appends CelebA, so
peak memory stays at one representation and neither session needs the other's features.

Two protocol changes from the earlier CelebA run, both of which invalidate its caches by design:

* **full training split**, not the 50k subsample. Waterbirds used 100% of its train split while
  CelebA used 31% -- an asymmetry with no scientific justification, only a compute one. It also
  left the minority group with ~425 distinct examples, which GroupDRO memorised.
* **model selection** on held-out worst-group accuracy, the protocol standard this study was
  missing.

**This exceeds one Colab session (~5.5 h of training) and that is deliberate.** Trimming epochs to
fit would be shrinking the experiment to fit the tooling. The run is resumable at arm granularity:
each finished arm is cached to Drive, so on a second pass it is a cache hit and only the remaining
arms train. Run cell 9 again after a disconnect until it reports 9 built arms.

## 0. Parameters -- **EDIT THESE**

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # needs kaggle.json in /content
CELEBA_DRIVE  = ""

# --- model selection: the protocol this study was missing --------------------------
# Sagawa et al. (2020), Kirichenko et al. (2023) and Liu et al. (2021) all select the checkpoint
# on worst-group VALIDATION accuracy, because these objectives overfit the minority group. Taking
# the last epoch instead let GroupDRO on CelebA drive train worst-group accuracy to 0.978 while
# evaluation stayed at ERM's level. Part of cache_key: which checkpoint is kept IS part of the
# representation, so turning this off gives different files rather than silently reusing these.
SELECT_BY          = "val_worst_group"
VAL_FRAC           = 0.1
VAL_MIN_PER_GROUP  = 15        # floor: 10% of Waterbirds' 56-example group is 6, SE ~0.20

FT_OBJECTIVES = ("erm", "groupdro", "reweight")
FT_HP = {
    "erm":      dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
    "reweight": dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
    "groupdro": dict(optimizer="sgd",  lr=1e-3, weight_decay=1e-2, groupdro_eta=0.05),
}
# Per (dataset, objective) and never shared: CelebA's train split is 34x Waterbirds', so one epoch
# is ~5.7 min there against seconds here.
FT_EPOCHS = {
    ("waterbirds", "erm"): 10, ("waterbirds", "reweight"): 10, ("waterbirds", "groupdro"): 20,
    ("celeba",     "erm"):  6, ("celeba",     "reweight"):  6, ("celeba",     "groupdro"):  6,
}
BATCH_SIZE    = 128
NUM_WORKERS   = None           # auto: cpu_count-1, capped at 12. Measured: no gain past ~8.
EXTRACT_BS    = 128            # fixed for the whole study; see finetune.py
EXTRACT_INFLIGHT = 3_500_000_000
AMP           = True           # fp32 weights/optimizer/loss, fp16 conv+matmul. In cache_key.
CACHE_DTYPE   = "float16"
N_SPLITS      = 10
HEADS         = ("erm", "dfr", "groupdro_ll")
SCORES        = ("APS", "RAPS", "THR")

# --- this notebook -----------------------------------------------------------------
RUN_DATASETS     = ("celeba",)       # stage 1 already built Waterbirds; this notebook only
                                     # trains CelebA and folds the stage-1 records in from CSV
FT_SEEDS         = (0, 1, 2, 3, 4)   # Waterbirds seeds, for reference only
CELEBA_SEEDS     = (0, 1, 2)
EXPECT_MULTI_SESSION = True    # ~5.5 h of training. Rather than trim epochs to fit a 4 h session,
                               # run this notebook twice: every finished arm is cached to Drive and
                               # is a cache hit on the second pass.
CELEBA_MAX_TRAIN = None        # FULL train split. The 50k subsample was compute-driven with no
                               # scientific justification, and it left GroupDRO only ~425 distinct
                               # minority examples to memorise. Full train gives ~1,390.

## 1. Drive + repo

In [ ]:
import os, sys, time, subprocess

def sh(cmd, check=True):
    """Show failures. A silent helper is how a Drive error becomes a wrong result three cells on."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"[shell FAILED rc={r.returncode}] {cmd}")
        if r.stderr.strip(): print(r.stderr[-2000:])
        if check: raise RuntimeError(f"command failed: {cmd}")
    return r.returncode == 0

def init_drive(mount="/content/drive", retries=3):
    """Mount Drive and PROVE it serves I/O -- the mount point can exist while every write fails."""
    from google.colab import drive
    for attempt in range(1, retries + 1):
        try:
            drive.mount(mount, force_remount=attempt > 1)
            os.makedirs(DRIVE_CACHE, exist_ok=True)
            probe, tok = os.path.join(DRIVE_CACHE, ".mount_probe"), str(time.time())
            with open(probe, "w") as fh: fh.write(tok)
            with open(probe) as fh: got = fh.read()
            os.remove(probe)
            if got != tok: raise IOError("probe read-back mismatch")
            free = os.statvfs(mount).f_bavail * os.statvfs(mount).f_frsize / 1e9
            print(f"Drive OK (attempt {attempt}) | ~{free:.1f} GB free")
            return
        except Exception as e:
            print(f"[drive] attempt {attempt}/{retries}: {e}"); time.sleep(5 * attempt)
    raise RuntimeError("Drive would not mount. Runtime > Disconnect and delete runtime, retry.")

init_drive()
REPO_DIR = "/content/vgscp"
sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

_cpus = os.cpu_count() or 4
if NUM_WORKERS is None:
    NUM_WORKERS = max(2, min(12, _cpus - 1))
    print(f"NUM_WORKERS auto -> {NUM_WORKERS} ({_cpus} vCPUs)")
elif NUM_WORKERS > _cpus:
    print(f"[warn] NUM_WORKERS {NUM_WORKERS} > {_cpus} vCPUs; clamping"); NUM_WORKERS = _cpus
print("repo:", os.getcwd())

## 2. Drive-backed caches

In [ ]:
os.makedirs("results", exist_ok=True)
for c in ("cache_clip", "cache_resnet", "cache_frozen", "cache_finetune", "study"):
    tgt = f"{DRIVE_CACHE}/{c}"; os.makedirs(tgt, exist_ok=True)
    sh(f"rm -rf results/{c}"); sh(f"ln -s {tgt} results/{c}")
    probe = f"results/{c}/.link_probe"                       # prove the link resolves onto Drive
    with open(probe, "w") as fh: fh.write("ok")
    assert os.path.exists(f"{tgt}/.link_probe"), f"results/{c} does not resolve to Drive"
    os.remove(probe)
CKPT_DIR = "/content/ft_ckpt"; os.makedirs(CKPT_DIR, exist_ok=True)   # local: Drive-write churn
print("caches symlinked to Drive and verified")
free = os.statvfs("/content/drive").f_bavail * os.statvfs("/content/drive").f_frsize / 1e9
print(f"Drive free: {free:.1f} GB")

## 3. Datasets

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
print("WATERBIRDS_ROOT =", os.environ["WATERBIRDS_ROOT"], "| CelebA OK =", CELEBA_OK)

## 4. Gate 1 -- analysis machinery

In [ ]:
rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_representation"],
                    capture_output=True, text=True)
print(rc.stdout[-2500:])
if rc.stderr.strip(): print(rc.stderr[-1200:])
assert rc.returncode == 0, "analysis validator FAILED"

## 5. Definitions (no training)

In [ ]:
from study_robust_train.representation import (build_repr_griddata,
                                                 run_representation_streaming,
                                                 write_representation_csv,
                                                 records_from_representation_csv,
                                                 write_representation_md)
from IPython.display import Markdown, display

REPR_CSV = "results/study/representation_records.csv"

def cfg_for(dataset, max_train=None):
    base = {"finetune": {"device": "cuda", "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
                         "amp": AMP, "max_train": max_train, "cache_dtype": CACHE_DTYPE,
                         "cache_dir": "results/cache_finetune", "ckpt_dir": CKPT_DIR,
                         "extract_batch_size": EXTRACT_BS,
                         "extract_inflight_bytes": EXTRACT_INFLIGHT,
                         "select_by": SELECT_BY, "val_frac": VAL_FRAC,
                         "val_min_per_group": VAL_MIN_PER_GROUP}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
    return base

def keys_for(dataset, seeds):
    return [(dataset, o, s) for o in FT_OBJECTIVES for s in seeds]

def builder(dataset, max_train=None):
    cfg = cfg_for(dataset, max_train)
    def build(ds, obj, seed):
        t = time.time()
        gd = build_repr_griddata(ds, obj, cfg, ft_seed=seed,
                                 epochs=FT_EPOCHS[(ds, obj)], **FT_HP.get(obj, {}))
        print(f"[built] {ds}/{obj}/s{seed}  ({(time.time()-t)/60:.1f} min)", flush=True)
        return gd
    return build

def show_plan(dataset, seeds, max_train=None):
    print(f"[plan] {dataset}: "
          + ", ".join(f"{o}x{len(seeds)}@{FT_EPOCHS[(dataset,o)]}ep" for o in FT_OBJECTIVES)
          + f" | max_train={max_train} | select_by={SELECT_BY!r}")

print("defined -- no training yet")

## 6. Gate 2 -- pre-flight on the LIVE config\n\nAudits the values actually in scope, prints the predicted cache keys, and reports how many arms are already cached -- a miss means training, which on a CPU runtime is catastrophically slow with no warning.

In [ ]:
from study_robust_train.preflight_representation import audit
assert SELECT_BY, ("SELECT_BY is empty: that reverts to the last-epoch protocol this revision "
                   "exists to fix. Set it back to 'val_worst_group'.")
assert audit(globals()), "PRE-FLIGHT FAILED -- fix the config above before spending GPU time"

## 7. CelebA availability

`CELEBA_OK` is set in the datasets cell; this re-checks and re-runs the preparation with full
output, so a failure names its own cause instead of asserting later.

In [ ]:
import json as _json
for c in ("kaggle.json", "/content/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json")):
    if os.path.exists(c):
        try:
            print(f"FOUND {c} | username={_json.load(open(c)).get('username','?')!r}")
        except Exception as e:
            print(f"*** {c} is not valid JSON: {e}")
        break
else:
    print("*** upload kaggle.json into /content (Files panel). Note /content does not survive a "
          "runtime restart.")

CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
print("CelebA OK =", CELEBA_OK, "|", CELEBA_ROOT)

## 8. Load the Waterbirds records from stage 1

Records, not features: the whole Waterbirds half is a few MB of CSV, so folding it in costs nothing
and the combined report covers both datasets.

In [ ]:
assert os.path.exists(REPR_CSV), (
    f"{REPR_CSV} missing -- run GPU stage 1 first; it writes the Waterbirds half.")
wb_records = records_from_representation_csv(REPR_CSV)
ds_present = sorted({r["dataset"] for r in wb_records})
print(f"loaded {len(wb_records)} records | datasets: {ds_present}")
assert "waterbirds" in ds_present, "stage 1 records do not contain waterbirds"
if "celeba" in ds_present:
    print("[note] the CSV already has CelebA rows; they will be REPLACED by this run")
    wb_records = [r for r in wb_records if r["dataset"] != "celeba"]
    print("       kept", len(wb_records), "waterbirds rows")

## 9. CelebA, full training split (~2.5-3.5 h)

Resumable: a completed arm is cached to Drive and never recomputed, so a lost session costs at most
the arm in flight. Watch `selected epoch k/N` -- if GroupDRO now selects an early epoch and its
eval worst-group accuracy rises above ERM's, the earlier collapse was the missing model selection
rather than the subsample.

In [ ]:
assert CELEBA_OK, "CelebA unavailable -- see cell 7"
show_plan("celeba", CELEBA_SEEDS, CELEBA_MAX_TRAIN)
out = run_representation_streaming(keys_for("celeba", CELEBA_SEEDS),
                                   builder("celeba", CELEBA_MAX_TRAIN),
                                   heads=HEADS, scores=SCORES, n_splits=N_SPLITS,
                                   prior_records=wb_records)
print()
print("combined records:", len(out["records"]), "| failed:", out["failed"])

## 10. Combined report

In [ ]:
write_representation_csv(out["records"], REPR_CSV)
display(Markdown(write_representation_md(out, "REPRESENTATION.md")))
print()
print("persisted:", f"{DRIVE_CACHE}/study/representation_records.csv")

## 11. Free the stale 50k caches

The subsampled CelebA arms are superseded. They are ~7 GB and nothing references them any more, so
removing them keeps Drive inside budget for the grid stage.

In [ ]:
import glob
stale = [p for p in glob.glob("results/cache_finetune/*celeba*mt50000*") ]
print(f"{len(stale)} superseded 50k file(s), "
      f"{sum(os.path.getsize(p) for p in stale)/1e9:.2f} GB")
for p in stale: print("  ", os.path.basename(p)[:96])
# Deliberately NOT deleted automatically -- inspect the list, then uncomment:
# for p in stale: os.remove(p)
# print("removed")